# 03b — Northern BC: connect the proposed IPCAs

Crops the aligned stack to the buffered bounding box of the **4 draft protected areas**
(`proposed_pa_v2_northern_bc.shp`), **locks them in as anchors**, and **up-weights connectivity**
(`transboundary_connectivity` + `climate_corridors` ×5) so the solve identifies the best land to
connect them. Parameters live in `config.ANALYSES["north_bc"]`; outputs →
`output_data/iter6_north_bc/`.

**Note:** the anchors are a large share of the small window, so `budget_pct` must exceed the
locked fraction (see the "locked-in / budget" line in cell 3) — tune it in config if the
feasibility guard stops the run. **Kernel:** `R (y2y)`. Ethan runs cell-by-cell.

In [ ]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
source("prioritizr_core.R")            # pr_* functions (crop/mask, lock-in, weights, solve)
ANALYSIS <- "north_bc"       # <-- the ONLY line that differs between 03a / 03b / 03c
PROJ <- normalizePath(getwd())         # run from the project root

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

In [ ]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

In [ ]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

In [ ]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

In [ ]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

In [ ]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

In [ ]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

In [ ]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

In [ ]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)